# Day 031 — Exercise 2: DeadLetterQueue

**What you'll build:** `DeadLetterQueue` — a container for failed items that exhausted all retries. `add(item, error)` stores the failed item with an error message and timestamp. `drain()` returns all stored items and resets the queue to empty. `peek()` returns items without draining. `size()` returns the current count.

**Why it matters:** When retries are exhausted, the item must go somewhere. Silently discarding it loses data. Crashing the batch stops processing. The DLQ routes failures to a holding area for later inspection, replay, or alert.

In [ ]:
from datetime import datetime

## Your Implementation

In [ ]:
class DeadLetterQueue:
    """
    Container for items that failed all retry attempts.

    Each record has: item (original value), error (str), context (dict),
    added_at (ISO timestamp).
    """

    def __init__(self):
        # TODO: self._items = []
        pass

    def add(self, item, error: str, context: dict | None = None) -> None:
        # TODO: append dict with item, error, context (or {}), added_at (ISO str)
        pass

    def drain(self) -> list:
        # TODO: atomic swap: items, self._items = self._items, []; return items
        pass

    def peek(self) -> list:
        # TODO: return list(self._items) — copy, not reference
        pass

    def size(self) -> int:
        # TODO: return len(self._items)
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: class defined with required methods
    try:
        assert 'DeadLetterQueue' in globals()
        for m in ('add', 'drain', 'peek', 'size'):
            assert hasattr(DeadLetterQueue, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: DeadLetterQueue with add/drain/peek/size')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    dlq = DeadLetterQueue()

    # Check 2: add increases size; record has required keys
    try:
        dlq.add('url_1', 'ConnectionTimeout')
        dlq.add(42,      'ValueError: bad id')
        assert dlq.size() == 2, f'size should be 2, got {dlq.size()}'
        passed += 1; print('\u2705 Check 2: add increases size to 2')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: record has correct keys and values
    try:
        items = dlq.peek()
        rec = items[0]
        for k in ('item', 'error', 'context', 'added_at'):
            assert k in rec, f'record missing key: {k}'
        assert rec['item']  == 'url_1',             f"item wrong: {rec['item']!r}"
        assert rec['error'] == 'ConnectionTimeout', f"error wrong: {rec['error']!r}"
        assert isinstance(rec['added_at'], str) and rec['added_at'], \
            f'added_at should be non-empty str: {rec["added_at"]!r}'
        passed += 1; print('\u2705 Check 3: record has correct keys and values')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: peek does NOT empty the queue
    try:
        before = dlq.size()
        _ = dlq.peek()
        after = dlq.size()
        assert before == after, \
            f'peek should not change size: before={before}, after={after}'
        passed += 1; print('\u2705 Check 4: peek is non-destructive')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: drain returns all items and empties the queue
    try:
        drained = dlq.drain()
        assert len(drained) == 2, f'should drain 2 items, got {len(drained)}'
        assert dlq.size() == 0, f'queue should be empty after drain, got {dlq.size()}'
        # Drain again → empty list
        again = dlq.drain()
        assert again == [], f'second drain should return [], got {again}'
        passed += 1; print('\u2705 Check 5: drain returns all items and empties queue')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
from datetime import datetime


class DeadLetterQueue:
    def __init__(self):
        self._items: list = []

    def add(self, item, error: str, context: dict | None = None) -> None:
        self._items.append({
            "item":     item,
            "error":    error,
            "context":  context or {},
            "added_at": datetime.now().isoformat(),
        })

    def drain(self) -> list:
        items, self._items = self._items, []
        return items

    def peek(self) -> list:
        return list(self._items)

    def size(self) -> int:
        return len(self._items)
```

</details>